[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_QuestionAnswering.ipynb)

# Benchmark: Question Answering

Scores a pretrained extractive QA model's exact-match and token-overlap F1 against gold-labeled
data, using `sparknlp.benchmark.Benchmark.evaluate(..., task="questionanswering")`.

**Dataset**: [SQuAD v1.1](https://rajpurkar.github.io/SQuAD-explorer/) dev split -- the standard
extractive-QA benchmark (Rajpurkar et al., 2016), CC BY-SA 4.0. We sample the first 300 questions
to keep this notebook quick to run; swap in the full dev set for a real benchmark.

**Model**: `BertForQuestionAnswering.pretrained()` (default: `bert_base_cased_qa_squad2`),
trained on SQuAD2.

> **Every reference answer is kept.** `label` is an `array<string>` of all the answers
> SQuAD lists for a question, and `Benchmark.evaluate` scores each prediction against its
> best-matching reference, the way the official SQuAD eval script does. Scoring against
> only the first answer would put EM/F1 several points below the model's real SQuAD number.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import MultiDocumentAssembler
from sparknlp.annotator import BertForQuestionAnswering
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import json
import urllib.request

url = "https://raw.githubusercontent.com/rajpurkar/SQuAD-explorer/master/dataset/dev-v1.1.json"
squad = json.loads(urllib.request.urlopen(url, timeout=60).read())

rows = []
for article in squad["data"]:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            if qa["answers"]:
                # SQuAD dev gives ~3 human answers per question on purpose: annotators disagree
                # on span boundaries ("1858" vs "in 1858", "Denver Broncos" vs "the Denver
                # Broncos"). The official eval script scores a prediction against every one of
                # them and keeps the best, so we pass them all. Keeping only answers[0] would
                # mark genuinely correct predictions wrong and report EM/F1 several points below
                # the model's real SQuAD score.
                answers = list(dict.fromkeys(a["text"] for a in qa["answers"]))
                rows.append((qa["question"], context, answers))
        if len(rows) >= 300:
            break
    if len(rows) >= 300:
        break

gold_data = spark.createDataFrame(rows, ["question", "context", "label"])
print(gold_data.count(), "questions")
gold_data.show(3, truncate=60)

303 questions
+----------------------------------------------------+------------------------------------------------------------+-----------------------+
|                                            question|                                                     context|                  label|
+----------------------------------------------------+------------------------------------------------------------+-----------------------+
|Which NFL team represented the AFC at Super Bowl 50?|Super Bowl 50 was an American football game to determine ...|         Denver Broncos|
|Which NFL team represented the NFC at Super Bowl 50?|Super Bowl 50 was an American football game to determine ...|      Carolina Panthers|
|                 Where did Super Bowl 50 take place?|Super Bowl 50 was an American football game to determine ...|Santa Clara, California|
+----------------------------------------------------+------------------------------------------------------------+-----------------------+
only s

## 2. Build the pipeline

> **Note: match `setCaseSensitive` to the checkpoint.** `bert_base_cased_qa_squad2` is
> (per its own name) a *cased* checkpoint -- leaving `caseSensitive` at its default mismatches
> its tokenization and silently degrades every prediction, so we set it explicitly here.

In [11]:
document_assembler = MultiDocumentAssembler() \
    .setInputCols(["question", "context"]) \
    .setOutputCols(["document_question", "document_context"])
span_classifier = BertForQuestionAnswering.pretrained() \
    .setInputCols(["document_question", "document_context"]).setOutputCol("answer") \
    .setCaseSensitive(True)

pipeline = Pipeline(stages=[document_assembler, span_classifier])
pipeline_model = pipeline.fit(gold_data)

bert_base_cased_qa_squad2 download started this may take some time.
Approximate size to download 384.7 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[OK!]

## 3. Run the benchmark

In [13]:
report = Benchmark.evaluate(
    pipeline_model, gold_data, task="questionanswering", text_col="context", label_col="label")
print(report)

questionanswering accuracy (n=303): exactMatch=0.3036, f1=0.3563